# Archivo de funciones y código

Básicamente aquí voy a ir poniendo babosadas que vaya programando para no colisionar con nadie más.

In [90]:
from scipy.optimize import minimize
from guppylang import guppy
from guppylang.defs import GuppyFunctionDefinition
from guppylang.std.builtins import comptime, result
from guppylang.std.angles import angle
from guppylang.std.quantum import h, measure, qubit, ry

In [91]:
def make_circuit(theta: float, pauli: str) -> GuppyFunctionDefinition:
    theta = float(theta[0])
    if pauli == "X":
        @guppy
        def circuit() -> None:
            q = qubit()
            ry(q, angle(comptime(theta)))
            h(q)
            result("m", measure(q))
    elif pauli == "Y":
        @guppy
        def circuit() -> None:
            q = qubit()
            ry(q, angle(comptime(theta)))
            sdg(q)
            h(q)
            result("m", measure(q))
    elif pauli == "Z":
        @guppy
        def circuit() -> None:
            q = qubit()
            ry(q, angle(comptime(theta)))
            result("m", measure(q))
    else:
        raise ValueError(f"Unsupported Pauli matrix: {pauli!r}")

    return circuit

In [92]:
def expval(theta: float, pauli: str, n_shots: int = 4000) -> float:
    if pauli == "I":
        return 1.0
    circuit = make_circuit(theta, pauli)
    shots = circuit.emulator(n_qubits=1).with_shots(n_shots).run()
    counts = shots.register_counts()["m"]
    n0, n1 = counts.get("0", 0), counts.get("1", 0)
    return (n0 - n1) / (n0 + n1)

In [93]:
def energy(theta: float, hamiltonian: list[tuple[float, str]], n_shots: int = 4000) -> float:
    return sum(
        coeff * expval(theta, pauli, n_shots) for coeff, pauli in hamiltonian
    )

In [95]:
ham = [
    (1.0, 'X'),
    (1.0, 'Z')
]
e = energy([0.0], ham, n_shots=100_000)

In [96]:
def cost(theta):
    ham = [
        (1.0, 'X'),
        (1.0, 'Z')
    ]
    return energy(theta, ham, n_shots = 10_000)

In [97]:
result = minimize(cost, 0.1, method='COBYLA')

In [98]:
result

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.4373999999999998
       x: [ 1.250e+00]
    nfev: 26
   maxcv: 0.0

In [100]:
1.250*np.pi

3.9269908169872414